[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pinecone-io/examples/blob/master/docs/quick-tour/hello-pinecone.ipynb) [![Open nbviewer](https://raw.githubusercontent.com/pinecone-io/examples/master/assets/nbviewer-shield.svg)](https://nbviewer.org/github/pinecone-io/examples/blob/master/docs/quick-tour/hello-pinecone.ipynb)

# Hello, Pinecone!

This notebook will walk through the steps to get a simple Pinecone index up and running.


## Prerequisites

First we need to install a few dependencies

In [6]:
!pip install -qU pandas==2.2.3 pinecone==9.1.0

import os
import random
import time

import pandas as pd
from pinecone import AwsRegion, CloudProvider, Metric, Pinecone, ServerlessSpec

## Getting started

We begin by instantiating an instance of the Pinecone client. To do this we need a [free API key](https://app.pinecone.io).

In [9]:
# Get your API key at app.pinecone.io
api_key = os.environ.get("PINECONE_API_KEY") or "pcsk_4N7sJ5_R8ockKTcYP22zkM8meY6pWXiBtC64PhJqUfAynh1ymWroichJ5jt5vvqjtB6TLW"

# Instantiate the Pinecone client
pc = Pinecone(
    # You can remove this for your own projects!
    api_key=api_key,
    source_tag="pinecone_examples:docs:quick_tour:hello_pinecone",
)

## Creating an index

With Pinecone you can create a vector index where you can store and search through your vector embeddings.

In [10]:
# Giving our index a name
index_name = "hello-pinecone"

In [11]:
# Delete the index if an index of the same name already exists
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

### Creating a Pinecone Index

When creating the index we need to define several configuration properties.

- `name` can be anything we like. The name is used as an identifier for the index when performing other operations such as `describe_index`, `delete_index`, and so on.
- `metric` specifies the similarity metric that will be used later when you make queries to the index.
- `dimension` should correspond to the dimension of the dense vectors produced by your embedding model. In this quick start, we are using made-up data so a small value is simplest.
- `spec` holds a specification which tells Pinecone how you would like to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/guides/index-data/create-an-index#cloud-regions).

There are more configurations available, but this minimal set will get us started.

In [12]:
pc.create_index(
    name=index_name,
    metric=Metric.COSINE,
    dimension=3,
    spec=ServerlessSpec(cloud=CloudProvider.AWS, region=AwsRegion.US_EAST_1),
)

IndexModel(name='hello-pinecone', metric='cosine', status=IndexStatus(ready=True, state='Ready'), spec=IndexSpec(serverless=ServerlessSpecInfo(cloud='aws', region='us-east-1', read_capacity={'mode': 'OnDemand', 'status': {'state': 'Ready', 'current_shards': None, 'current_replicas': None}}, source_collection=None, schema=None), pod=None, byoc=None), host='https://hello-pinecone-d86tzbr.svc.aped-4627-b74a.pinecone.io', private_host=None, vector_type='dense', dimension=3, deletion_protection='disabled', tags=None, embed=None, created_at=None)

We can look up the configuration for the index anytime we like by using `describe_index`

In [ ]:
description = pc.describe_index(name=index_name)
description

## Upserting data into the index

We can see the index ready. Now we will create some simple vectors that will serve as our examples.

In [ ]:
# Instantiate an Index client
index = pc.Index(host=description.host)

In [ ]:
# Set seed for reproducible results
random.seed(42)


def create_simulated_data_in_df(num_vectors):
    df = pd.DataFrame(
        data={
            "id": [f"id-{i}" for i in range(num_vectors)],
            "vector": [
                [random.random() for _ in range(description.dimension)]
                for _ in range(num_vectors)
            ],
        }
    )
    return df


df = create_simulated_data_in_df(10)

df.head()

We perform `upsert` operations in our index. This call will insert a new vector in the index or update the vector if the id was already present.

In [ ]:
index.upsert(vectors=zip(df.id, df.vector))  # insert vectors

In [ ]:
# Wait for vectors to be indexed and available for querying
while index.describe_index_stats().total_vector_count == 0:
    time.sleep(1)

print("Vectors are ready for querying!")

In [ ]:
# View index stats
index.describe_index_stats()

## Running a query

Next we can run a query.

In a more realistic scenario, the `vector` values passing into `query` would be an embedding vector of something meaningful. But for this simple walkthrough we will use made up values. The query will succeed as long as the dimension matches the dimension of our index.

`top_k` specifies the number of results we would like returned. The method will return up to `top_k` results, but may be less if there are fewer than `top_k` vectors in your index or if all indexes have been filtered out using metadata filters.

In [ ]:
# In a more realistic scenario, this would be an embedding vector
# that encodes something meaningful. For this simple demo, we will
# make up a vector that matches the dimension of our index.
query_embedding = [2.0, 2.0, 2.0]

index.query(vector=query_embedding, top_k=5, include_values=True)

## Delete the Index
Delete the index once you are sure that you do not want to use it anymore. **Deletion is permanent**. Once the index is deleted, you cannot use it again.

In [ ]:
pc.delete_index(name=index_name)

## Next steps

Now that you've seen the basics, explore more:

- [Semantic search](https://docs.pinecone.io/guides/search/semantic-search) - Search with real embeddings
- [Metadata filtering](https://docs.pinecone.io/guides/search/filter-by-metadata) - Filter results by metadata
- [RAG Getting Started](https://github.com/pinecone-io/examples/blob/main/docs/rag-getting-started.ipynb) - Build a retrieval-augmented generation app